# PolyGen on Google Colab (Week 1 + Week 2)

Runs the real pipeline on a free Colab GPU: **prompt -> Shap-E mesh -> clean -> xatlas UV unwrap -> Stable Diffusion + ControlNet texture -> baked textured model -> 3D preview**.

**How to use**
1. Menu **Runtime > Change runtime type > T4 GPU**, then **Save**.
2. Run the cells top to bottom (Shift+Enter). When asked, upload the project ZIP `PolyGen-Text-to-3D-Asset-Pipeline.zip`.
3. The first generation downloads several GB of model weights to Colab (not to your computer). Expect 5-10 minutes the first time.

**Honest status**
- Cells 1-6 (GPU check, install, unit tests, Shap-E mesh) are the supported path.
- Cells 7-9 (texturing, validation, audit) follow the same code paths but depend on the Stable Diffusion / ControlNet downloads working in your session.
- The TripoSR cell is **optional and NOT verified in Colab**; it compiles native code and may need fixes.
- Colab sessions are temporary. Download your results (last cell) before the session ends.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 2. Get the project
Upload the ZIP when prompted. (Alternative: if the project is on GitHub, replace this cell with `!git clone <your-repo-url>` and set `ROOT` to the cloned folder.)

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

uploaded = files.upload()                      # choose PolyGen-Text-to-3D-Asset-Pipeline.zip
zipfile.ZipFile(next(iter(uploaded))).extractall('/content')
ROOT = next(Path('/content').glob('*/backend/app')).parents[1]
print("Project root:", ROOT)
print(sorted(p.name for p in ROOT.iterdir()))

## 3. Install dependencies
PyTorch is already installed on Colab, so it is **not** reinstalled. If pip asks you to *restart the session* afterwards, do that, then continue from cell 4 (cell 2 does not need to be rerun).

In [ ]:
!pip install -q -r {ROOT}/backend/requirements.txt

In [ ]:
import diffusers, transformers, trimesh, xatlas
print("diffusers", diffusers.__version__, "| transformers", transformers.__version__, "| trimesh", trimesh.__version__)

## 4. Run the unit tests
Tests for features whose dependencies are installed should PASS; any SKIPPED test names the missing dependency.

In [ ]:
!cd {ROOT} && python -m unittest discover -s tests -t . -v

## 5. Route A - Shap-E: text -> 3D mesh (geometry only)
Change `PROMPT` and rerun. `texture='none'` skips Stable Diffusion, so this is the fastest check.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'backend'))
from app.config import settings
from app.pipeline import run_pipeline

PROMPT = "a worn wooden tavern chair"
JOB_MESH = settings.output_dir / 'colab_mesh'

result = run_pipeline(PROMPT, JOB_MESH, generator='shape', seed=0, texture='none',
                      progress=lambda s: print('->', s))
print(result['stats'])

In [ ]:
import base64, trimesh
from IPython.display import HTML

def show_glb(obj_path, height=480):
    """Convert an OBJ (with MTL/texture if present) to GLB and show it in an interactive viewer."""
    obj_path = Path(obj_path)
    scene = trimesh.load(str(obj_path), process=False)
    glb = obj_path.with_suffix('.glb')
    scene.export(str(glb))
    b64 = base64.b64encode(glb.read_bytes()).decode()
    return HTML('<script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js"></script>'
                '<model-viewer src="data:model/gltf-binary;base64,' + b64 + '" camera-controls auto-rotate '
                'shadow-intensity="1" style="width:100%;height:' + str(height) + 'px;background:#12151a"></model-viewer>')

show_glb(JOB_MESH / result['obj'])

## 6. Geometry check
The stats above should show a sensible face count and `components` close to 1. Shap-E meshes are rough and are sometimes blobby; try other prompts or seeds.

## 7. Texturing: Stable Diffusion + ControlNet (depth-guided)
Uses the same mesh route, then: xatlas UV unwrap -> depth render -> ControlNet image -> bake into the UV atlas.
Set `TEXTURE = 'flat'` for the simpler flat-material mode. If the GPU runs out of memory, use *Runtime > Restart session* and rerun cells 5 and 7 only.

In [ ]:
TEXTURE = 'controlnet'          # 'controlnet' | 'flat'
JOB_TEX = settings.output_dir / 'colab_textured'

result_tex = run_pipeline(PROMPT, JOB_TEX, generator='shape', seed=0, texture=TEXTURE,
                          progress=lambda s: print('->', s))
print(result_tex['stats'])

In [ ]:
from PIL import Image
from IPython.display import display
for name in ['depth.png', 'front_view.png', 'model_texture.png']:
    p = JOB_TEX / name
    if p.exists():
        print(name); display(Image.open(p).resize((320, 320)))

In [ ]:
show_glb(JOB_TEX / result_tex['obj'])

## 8. Validate the export
Checks the OBJ has UVs in [0,1], faces index them, and the MTL points at an existing texture file.

In [ ]:
!python {ROOT}/scripts/validate_export.py {JOB_TEX}

## 9. Geometry audit (optional, slow)
Generates 5 objects and writes metrics plus 4-view depth sheets. Takes several minutes.

In [ ]:
!python {ROOT}/scripts/geometry_audit.py --generator shape
audit = settings.output_dir / 'audit'
print((audit / 'report.md').read_text())
for sheet in sorted(audit.glob('*_views.png')):
    print(sheet.name); display(Image.open(sheet))

## 10. Route B - TripoSR image -> 3D (optional, NOT verified in Colab)
This route first makes an image with SD-Turbo, then runs TripoSR. It compiles `torchmcubes` and downloads more weights, so it may fail. If it errors, copy the message back for debugging; Route A is unaffected.

In [ ]:
!bash {ROOT}/scripts/setup_triposr.sh
!pip install -q -r {ROOT}/backend/requirements-triposr.txt

In [ ]:
JOB_TRIPO = settings.output_dir / 'colab_triposr'
result_tripo = run_pipeline(PROMPT, JOB_TRIPO, generator='triposr', seed=0, texture='none',
                            progress=lambda s: print('->', s))
print(result_tripo['stats'])
show_glb(JOB_TRIPO / result_tripo['obj'])

## 11. Download your results

In [ ]:
import shutil
shutil.make_archive('/content/polygen_results', 'zip', settings.output_dir)
files.download('/content/polygen_results.zip')